In [ ]:
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
from tpch import load_part, load_lineitem, load_supplier, load_orders, load_customer, load_nation, load_region, q08
DATA_ROOT = "/home/jupyter/tpch-dbgen/data" #"/home/dias-benchmarks/tpch/data"
STORAGE_OPTS = {}
part = load_part(DATA_ROOT, **STORAGE_OPTS)
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)
region = load_region(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%%time
q08(part, lineitem, supplier, orders, customer, nation, region)


In [3]:
%%time
def q08_cudf(part, lineitem, supplier, orders, customer, nation, region):
    # 1) PART → only “ECONOMY ANODIZED STEEL”
    parts_f = part[part.P_TYPE == "ECONOMY ANODIZED STEEL"][["P_PARTKEY"]]

    # 2) LINEITEM → select cols + compute VOLUME
    li = lineitem[[
        "L_PARTKEY","L_SUPPKEY","L_ORDERKEY",
        "L_EXTENDEDPRICE","L_DISCOUNT"
    ]]
    li["VOLUME"] = li.L_EXTENDEDPRICE * (1 - li.L_DISCOUNT)

    # 3) PART ⨝ LINEITEM
    total = parts_f.merge(
        li, left_on="P_PARTKEY", right_on="L_PARTKEY", how="inner"
    )[
        ["L_SUPPKEY","L_ORDERKEY","VOLUME"]
    ]

    # 4) SUPPLIER → get supplier’s nation key
    sup_n = supplier[["S_SUPPKEY","S_NATIONKEY"]]
    total = total.merge(
        sup_n, left_on="L_SUPPKEY", right_on="S_SUPPKEY", how="inner"
    )[
        ["L_ORDERKEY","VOLUME","S_NATIONKEY"]
    ]

    # 5) ORDERS → filter by date, extract year
    start, end = pd.Timestamp("1995-01-01"), pd.Timestamp("1997-01-01")
    ord_f = orders[
        (orders.O_ORDERDATE >= start) & (orders.O_ORDERDATE < end)
    ][["O_ORDERKEY","O_CUSTKEY","O_ORDERDATE"]]
    ord_f["O_YEAR"] = ord_f.O_ORDERDATE.dt.year
    ord_f = ord_f[["O_ORDERKEY","O_CUSTKEY","O_YEAR"]]

    total = total.merge(
        ord_f, left_on="L_ORDERKEY", right_on="O_ORDERKEY", how="inner"
    )[
        ["VOLUME","S_NATIONKEY","O_CUSTKEY","O_YEAR"]
    ]

    # 6) CUSTOMER → get customer’s nation key
    cust_f = customer[["C_CUSTKEY","C_NATIONKEY"]]
    total = total.merge(
        cust_f, left_on="O_CUSTKEY", right_on="C_CUSTKEY", how="inner"
    )[
        ["VOLUME","S_NATIONKEY","O_YEAR","C_NATIONKEY"]
    ]

    # 7) NATION (customer) → map to region
    n1 = nation[["N_NATIONKEY","N_REGIONKEY"]]
    total = total.merge(
        n1, left_on="C_NATIONKEY", right_on="N_NATIONKEY", how="inner"
    )[
        ["VOLUME","S_NATIONKEY","O_YEAR","N_REGIONKEY"]
    ]

    # 8) NATION (supplier) → get supplier’s nation name
    n2 = nation[["N_NATIONKEY","N_NAME"]].rename(columns={"N_NAME":"NATION"})
    total = total.merge(
        n2, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner"
    )[
        ["VOLUME","O_YEAR","N_REGIONKEY","NATION"]
    ]

    # 9) REGION → only “AMERICA”
    reg_f = region[region.R_NAME == "AMERICA"][["R_REGIONKEY"]]
    total = total.merge(
        reg_f, left_on="N_REGIONKEY", right_on="R_REGIONKEY", how="inner"
    )[
        ["VOLUME","O_YEAR","NATION"]
    ]

    # 10) compute Brazil share in one groupby
    total["VOLUME_BRAZIL"] = total.VOLUME.where(total.NATION == "BRAZIL", 0)
    agg = total.groupby("O_YEAR", sort=True).agg(
        TOTAL_VOLUME  = ("VOLUME",        "sum"),
        BRAZIL_VOLUME = ("VOLUME_BRAZIL", "sum"),
    )
    agg["MKT_SHARE"] = agg.BRAZIL_VOLUME / agg.TOTAL_VOLUME

    return agg.reset_index()[["O_YEAR","MKT_SHARE"]]
q08_cudf(part, lineitem, supplier, orders, customer, nation, region)

CPU times: user 152 ms, sys: 50.3 ms, total: 202 ms
Wall time: 190 ms


,O_YEAR,MKT_SHARE
0,1995,0.034436
1,1996,0.041486
